## 넷플릭스 컨텐츠 분석

- 넷플릭스에 입점된 컨텐츠 목록(netflix.csv)을 받고,
- 전처리 한 후
- 차트화 하기(그래프는 취향껏)

In [139]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
netflix = pd.read_csv('C:/Users/user/Desktop/VSCode/1209_1210통계분석/netflix_titles.csv')
netflix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       7787 non-null   object
 1   type          7787 non-null   object
 2   title         7787 non-null   object
 3   director      5398 non-null   object
 4   cast          7069 non-null   object
 5   country       7280 non-null   object
 6   date_added    7777 non-null   object
 7   release_year  7787 non-null   int64 
 8   rating        7780 non-null   object
 9   duration      7787 non-null   object
 10  listed_in     7787 non-null   object
 11  description   7787 non-null   object
dtypes: int64(1), object(11)
memory usage: 730.2+ KB


In [4]:
netflix

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...
...,...,...,...,...,...,...,...,...,...,...,...,...
7782,s7783,Movie,Zozo,Josef Fares,"Imad Creidi, Antoinette Turk, Elias Gergi, Car...","Sweden, Czech Republic, United Kingdom, Denmar...","October 19, 2020",2005,TV-MA,99 min,"Dramas, International Movies",When Lebanon's Civil War deprives Zozo of his ...
7783,s7784,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,"March 2, 2019",2015,TV-14,111 min,"Dramas, International Movies, Music & Musicals",A scrappy but poor boy worms his way into a ty...
7784,s7785,Movie,Zulu Man in Japan,NaN,Nasty C,NaN,"September 25, 2020",2019,TV-MA,44 min,"Documentaries, International Movies, Music & M...","In this documentary, South African rapper Nast..."
7785,s7786,TV Show,Zumbo's Just Desserts,NaN,"Adriano Zumbo, Rachel Khoo",Australia,"October 31, 2020",2019,TV-PG,1 Season,"International TV Shows, Reality TV",Dessert wizard Adriano Zumbo looks for the nex...


#### 어떻게 전처리할까?

- type = Movie 설정
- country별로
- duration 길이가 얼마만한지 비교 (1)
- date_added 받아서 월별로 출시한 영화 몇 개인지 비교 (2)
  - -> 두 비교를 통해 언제 영화를 출시해야 유리할 지 추정

In [15]:
netflix.shape #7787개의 데이터와 12가지의 데이터 종류 표현

(7787, 12)

In [85]:
netflix_df = netflix.loc[netflix['type'] == "Movie"]

#NaN 값 처리
netflix_df = netflix_df.dropna()
netflix_df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...
6,s7,Movie,122,Yasir Al Yasiri,"Amina Khalil, Ahmed Dawood, Tarek Lotfy, Ahmed...",Egypt,"June 1, 2020",2019,TV-MA,95 min,"Horror Movies, International Movies","After an awful accident, a couple admitted to ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7778,s7779,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
7780,s7781,Movie,Zoo,Shlok Sharma,"Shashank Arora, Shweta Tripathi, Rahul Kumar, ...",India,"July 1, 2018",2018,TV-MA,94 min,"Dramas, Independent Movies, International Movies",A drug dealer starts having doubts about his t...
7781,s7782,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."
7782,s7783,Movie,Zozo,Josef Fares,"Imad Creidi, Antoinette Turk, Elias Gergi, Car...","Sweden, Czech Republic, United Kingdom, Denmar...","October 19, 2020",2005,TV-MA,99 min,"Dramas, International Movies",When Lebanon's Civil War deprives Zozo of his ...


In [100]:
country = netflix_df.groupby('type')['country']
duration = netflix_df.groupby('type')['duration']
netflix_df['date_added'] = netflix_df['date_added'].astype(str).str.split(' ').str[0]
date = netflix_df.groupby('type')['date_added']

In [107]:
const = pd.concat([country.describe(), duration.describe(), date.describe()])
const

,count,unique,top,freq
type,,,,
Movie,4673,536,United States,1627
Movie,4673,186,93 min,121
Movie,4673,12,October,487


In [ ]:
#나라별로 분기
netflix_df["country"].drop_duplicates().sort_values().to_csv('country_list.csv', index=False, header=True, sep=',')

In [181]:
countries2

array(['Argentina', 'Brazil', 'France', 'Poland', 'Germany', 'Denmark',
       'Chile', 'Peru', 'United States', 'Qatar', 'Spain', 'Uruguay',
       'Serbia', 'Australia', 'Canada', 'Ireland', 'India', 'Iraq',
       'United Arab Emirates', 'United Kingdom', 'Austria',
       'Czech Republic', 'Bangladesh', 'Belgium', 'Netherlands',
       'Afghanistan', 'China', 'Colombia', 'Bulgaria', 'Cambodia',
       'Hungary', 'Japan', 'Luxembourg', 'Mexico', 'South Africa',
       'Nigeria', 'Norway', 'South Korea', 'Cayman Islands', 'Hong Kong',
       'Nepal', 'Morocco', 'Taiwan', 'Croatia', 'Slovenia', 'Montenegro',
       'Slovakia', 'Portugal', 'Sweden', 'Italy', 'Israel', 'Egypt',
       'Algeria', 'Finland', 'Latvia', '', 'Romania', 'Switzerland',
       'Iran', 'Lebanon', 'Malta', 'Singapore', 'Senegal', 'Georgia',
       'Jordan', 'Sri Lanka', 'Ghana', 'Guatemala', 'Iceland', 'Malaysia',
       'Pakistan', 'Soviet Union', 'Turkey', 'Indonesia', 'Philippines',
       'Greece', 'Albania',

In [250]:
countries = pd.read_csv('country_list.csv', header = None, skiprows=[0], encoding = 'utf-8')
countries2 = countries[0].str.replace('"', '').str.split(',').explode().str.strip()
countries2 = countries2.unique()
countries2 = pd.DataFrame(countries2)

country_index = {}
# for index, value in enumerate(countries2):
#     country_index = {
#         'index': index,
#         'value': value[index]
#     }
# country_index

countries3 = netflix_df.groupby("country")["duration"].count()


In [273]:
netflix_df2 = netflix_df.copy()
netflix_df3 = netflix_df2.groupby("country")[["duration"]].count()
netflix_df3 = netflix_df3.reset_index()
idx = netflix_df3.set_index('country').index.to_series()

country_index = {'country': netflix_df3, 'index': netflix_df3.index}
# netflix_df2 = netflix_df2.apply(country_index)
netflix_df2['country_idx'] = netflix_df2['country'].map(idx)
# print(netflix_df2[['country', 'country_idx']])
netflix_df2['country_num'] = netflix_df2['country'].astype('category').cat.codes
netflix_df2
# netflix_df2['countries2'] = netflix_df3.apply(lambda x: x.index)
# netflix_df2

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,country_idx,country_num
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,December,2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...,Mexico,236
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,December,2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow...",Singapore,293
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,November,2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi...",United States,433
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,January,2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...,United States,433
6,s7,Movie,122,Yasir Al Yasiri,"Amina Khalil, Ahmed Dawood, Tarek Lotfy, Ahmed...",Egypt,June,2019,TV-MA,95 min,"Horror Movies, International Movies","After an awful accident, a couple admitted to ...",Egypt,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7778,s7779,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,November,2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,United States,433
7780,s7781,Movie,Zoo,Shlok Sharma,"Shashank Arora, Shweta Tripathi, Rahul Kumar, ...",India,July,2018,TV-MA,94 min,"Dramas, Independent Movies, International Movies",A drug dealer starts having doubts about his t...,India,171
7781,s7782,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,January,2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",United States,433
7782,s7783,Movie,Zozo,Josef Fares,"Imad Creidi, Antoinette Turk, Elias Gergi, Car...","Sweden, Czech Republic, United Kingdom, Denmar...",October,2005,TV-MA,99 min,"Dramas, International Movies",When Lebanon's Civil War deprives Zozo of his ...,"Sweden, Czech Republic, United Kingdom, Denmar...",337


In [202]:
countries2

,0
0,Argentina
1,Brazil
2,France
3,Poland
4,Germany
...,...
99,Liechtenstein
100,Nicaragua
101,Venezuela
102,Vietnam


In [186]:
netflix_df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,December,2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,December,2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,November,2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,January,2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...
6,s7,Movie,122,Yasir Al Yasiri,"Amina Khalil, Ahmed Dawood, Tarek Lotfy, Ahmed...",Egypt,June,2019,TV-MA,95 min,"Horror Movies, International Movies","After an awful accident, a couple admitted to ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7778,s7779,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,November,2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
7780,s7781,Movie,Zoo,Shlok Sharma,"Shashank Arora, Shweta Tripathi, Rahul Kumar, ...",India,July,2018,TV-MA,94 min,"Dramas, Independent Movies, International Movies",A drug dealer starts having doubts about his t...
7781,s7782,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,January,2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."
7782,s7783,Movie,Zozo,Josef Fares,"Imad Creidi, Antoinette Turk, Elias Gergi, Car...","Sweden, Czech Republic, United Kingdom, Denmar...",October,2005,TV-MA,99 min,"Dramas, International Movies",When Lebanon's Civil War deprives Zozo of his ...


In [185]:
netflix_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4673 entries, 1 to 7783
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       4673 non-null   object
 1   type          4673 non-null   object
 2   title         4673 non-null   object
 3   director      4673 non-null   object
 4   cast          4673 non-null   object
 5   country       4673 non-null   object
 6   date_added    4673 non-null   object
 7   release_year  4673 non-null   int64 
 8   rating        4673 non-null   object
 9   duration      4673 non-null   object
 10  listed_in     4673 non-null   object
 11  description   4673 non-null   object
dtypes: int64(1), object(11)
memory usage: 474.6+ KB


In [183]:
heatmap = netflix_df[['country', 'duration', 'date_added']]
colormap = sns.diverging_palette(350, 245, s=60, as_cmap=True)
sns.heatmap(heatmap.astype(float).corr(), linewidths = 0.1, vmax = 1.0, square = True, cmap = colormap, linecolor = 'white', annot = True, annot_kws = {"size": 10})
plt.xticks(rotation = 45, fontsize = 10)
plt.show()

ValueError: could not convert string to float: 'Mexico'